In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score
from torchmetrics.functional.retrieval import retrieval_hit_rate, retrieval_precision


# Script to calculate NDKL metric
# References: Geyik, S. C., Ambler, S., & Kenthapadi, K. (2019, July).
# Fairness-aware ranking in search & recommendation systems with application to linkedin talent search.
# In Proceedings of the 25th acm sigkdd international conference on knowledge discovery & data mining (pp. 2221-2231).


def __kl_divergence(p, q):
    """
    Calculate KL-Divergence between P and Q, with epsilon to avoid divide by zero.
    :param p: Numpy array p distribution.
    :param q: Numpy array q distribution.
    :return: KL-Divergence score.
    """
    epsilon = 0.0000001  # Epsilon is used here to avoid P or Q is equal to 0. "
    p = p + epsilon
    q = q + epsilon

    return np.sum(p * np.log(p / q))


def NDKL(ranking_df, item_group_dict):
    """
    Calculate Normalized Discounted KL-Divergence Score (Geyik et al.).
    :param ranking_df: Pandas dataframe of ranking(s).
    :param item_group_dict: Dictionary of items (keys) and their group membership (values).
    :return: NDKL value.
    """
    if len(ranking_df.columns) > 1:
        raise AssertionError("NDKL can only be calculated on a single ranking.")

    single_ranking = ranking_df[ranking_df.columns[0]]  # isolate ranking
    single_ranking = np.array(
        single_ranking[~pd.isnull(single_ranking)]
    )  # drop any NaNs

    group_ids = [item_group_dict[c] for c in single_ranking]
    unique_grps = np.unique(group_ids)
    group_ids = np.asarray(
        [np.argwhere(unique_grps == grp_of_item)[0, 0] for grp_of_item in group_ids]
    )
    num_groups = np.max(group_ids)
    num_items = len(group_ids)

    dr = __distributions(group_ids, num_groups)  # Distributions per group
    Z = __Z_Vector(num_items)  # Array of Z scores

    # Eq. 4 in Geyik et al.
    return (1 / np.sum(Z)) * np.sum(
        [
            Z[i]
            * __kl_divergence(__distributions(group_ids[0 : i + 1], num_groups), dr)
            for i in range(0, num_items)
        ]
    )


def __distributions(ranking, num_groups):
    """
    Calculate the proportion of each group
    :param ranking: Numpy array of group id represented in the ranking.
    :param num_groups: Int, number of distinct groups
    :return: Numpy array of each group's proportion.
    """
    return np.array(
        [((ranking == i).sum()) / len(ranking) for i in range(0, num_groups + 1)]
    )


def __Z_Vector(k):
    """
    Calculate Z score
    :param k: Int, position of ranking.
    :return: Numpy array of Z values.
    """
    return 1 / np.log2(np.array(range(0, k)) + 2)


import pandas
import torch
import numpy as np
import torch
from torchmetrics.functional.retrieval import retrieval_hit_rate
import torch.nn.functional as F

def fair_metric(pred, labels, sens):
    idx_s0 = sens == 0
    idx_s1 = sens == 1
    idx_s0_y1 = np.bitwise_and(idx_s0, labels == 1)
    idx_s1_y1 = np.bitwise_and(idx_s1, labels == 1)
    parity = abs(sum(pred[idx_s0]) / sum(idx_s0) - sum(pred[idx_s1]) / sum(idx_s1))
    equality = abs(
        sum(pred[idx_s0_y1]) / sum(idx_s0_y1) - sum(pred[idx_s1_y1]) / sum(idx_s1_y1)
    )
    return parity.item(), equality.item()


def eval_hits(y_pred_pos, y_pred_neg, K, type_info):
        '''
            compute Hits@K
            For each positive target node, the negative target nodes are the same.

            y_pred_neg is an array.
            rank y_pred_pos[i] against y_pred_neg for each i
        '''

        if len(y_pred_neg) < K:
            return {'hits@{}'.format(K): 1.}

        if type_info == 'torch':
            kth_score_in_negative_edges = torch.topk(y_pred_neg, K)[0][-1]
            hitsK = float(torch.sum(y_pred_pos > kth_score_in_negative_edges).cpu()) / len(y_pred_pos)

        # type_info is numpy
        else:
            kth_score_in_negative_edges = np.sort(y_pred_neg)[-K]
            hitsK = float(np.sum(y_pred_pos > kth_score_in_negative_edges)) / len(y_pred_pos)

        return {'hits@{}'.format(K): hitsK}
    
    
def fairness_aware_topk(outputs, K, sens, type_info):
        '''
            compute Hits@K
            For each positive target node, the negative target nodes are the same.

            y_pred_neg is an array.
            rank y_pred_pos[i] against y_pred_neg for each i
        '''
        
        sens_stats = F.one_hot(sens, 2).float().sum(0) / len(sens)
        

        if len(outputs) < K:
            return {'hits@{}'.format(K): 1.}

        if type_info == 'torch':
            top_k = torch.topk(outputs, K).indices
            top_k_stats = F.one_hot(sens[top_k], 2).float().sum(0) / K
            sens_diff = torch.abs(top_k_stats - sens_stats).sum().item()
        else:
            top_k = np.argsort(outputs)[-K:]
            top_k_stats = F.one_hot(sens[top_k], 2).float().sum(0) / K
            sens_diff = np.abs(top_k_stats - sens_stats).sum()

        return sens_diff

    
def fairness_aware_hits(y_pred_pos, y_pred_neg, K, sens_pos, type_info):
        '''
            compute Hits@K
            For each positive target node, the negative target nodes are the same.

            y_pred_neg is an array.
            rank y_pred_pos[i] against y_pred_neg for each i
        '''
        
        sens_stats = F.one_hot(sens_pos, 2).float().sum(0) / len(sens_pos)
        

        if len(y_pred_neg) < K:
            return {'hits@{}'.format(K): 1.}

        if type_info == 'torch':
            
            kth_score_in_negative_edges = torch.topk(y_pred_neg, K)[0][-1]
            sens_pos_in_top_k = sens_pos[y_pred_pos > kth_score_in_negative_edges]
            sens_in_hits_at_k_stats = F.one_hot(sens_pos_in_top_k, 2).float().sum(0) / len(sens_pos_in_top_k)
            sens_diff = torch.abs(sens_in_hits_at_k_stats - sens_stats).sum().item()
        
        # type_info is numpy
        else:
            kth_score_in_negative_edges = np.sort(y_pred_neg)[-K]
            sens_pos_in_top_k = sens_pos[y_pred_pos > kth_score_in_negative_edges]
            sens_in_hits_at_k_stats = F.one_hot(sens_pos_in_top_k, 2).float().sum(0) / len(sens_pos_in_top_k)
            sens_diff = np.abs(sens_in_hits_at_k_stats - sens_stats).sum()

        return sens_diff
    
    
    
def edge_cosine_similarity(embeddings, edge_index):
    '''
        Compute the cosine similarity of the embeddings of the edges
    '''
    src, dst = edge_index
    src_emb = embeddings[src, :]
    dst_emb = embeddings[dst, :]
    cosine_similarity = torch.nn.functional.cosine_similarity(src_emb, dst_emb)
    return cosine_similarity


import matplotlib.pyplot as plt
import itertools
import random

def plot_coordinates(coords, labels=None, x_axis_label='X coordinate', y_axis_label='Y coordinate', title='Comparison of Different Methods'):
    def process_styles(labels):
        fair_model_styles = {
            'NIFTY' : 'D',
            'FairGNN': 'D',
            'FairVGNN': 'D',
            'GRAPHAIR': 'o',
            'EDITS': 's'
        }
        
        model_styles = {
            'GAE' : 'r',
            'NCN' : 'b',
            'SEAL' : 'g',
        }
        
        split_label_names = map(lambda x: x.split('_'), labels)
        
        for split_label in split_label_names:
            if len(split_label) != 2:
                raise ValueError("Labels must be in the format 'model_fairness'")
        
        return list(map(lambda x: (fair_model_styles[x[0]], model_styles[x[1]]), split_label_names))
        
        
    
    random.seed(42)

    try:
        styles = process_styles(labels)
    except Exception as e:
        print("Failed processing styles", e)
        # Available markers (symbols) for plotting
        markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', '1', '2', '3', '4', '8']
        colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']
        
        # Create combinations of markers and colors
        styles = list(itertools.product(markers, colors))
        random.shuffle(styles)
    
    # If labels are not provided, use default labels
    if labels is None:
        labels = [f'Method {i+1}' for i in range(len(coords))]
    
    # Ensure we have the same number of labels as coordinates
    if len(labels) != len(coords):
        raise ValueError("The number of labels must match the number of coordinate pairs.")
    
    # Create the plot
    plt.figure(figsize=(12, 8))
    
    # Plot each coordinate pair with a unique symbol and label
    for (x, y), label, style in zip(coords, labels, styles):
        marker, color = style
        plt.plot(x, y, marker=marker, color=color, markersize=10, label=label, linestyle='None')
    
    # Add labels and title
    plt.xlabel(x_axis_label)
    plt.ylabel(y_axis_label)
    plt.title(title)
    
    # Add a legend
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    # Display the grid
    plt.grid(True)
    
    # Adjust layout to prevent clipping of tick-labels
    plt.tight_layout()
    
    # Show the plot
    plt.show()
    

In [2]:
import numpy as np
import math
from collections import defaultdict as ddict
import torch
import torch.nn.functional as F
import copy
# References: Geyik, S. C., Ambler, S., & Kenthapadi, K. (2019, July).
# Fairness-aware ranking in search & recommendation systems with application to linkedin talent search.
# In Proceedings of the 25th acm sigkdd international conference on knowledge discovery & data mining (pp. 2221-2231).
import pandas as pd


def DETCONSTSORT(
    current_ranking_df, item_group_dict, current_ranking_scores_df, distribution, k
):
    """
    DetConstSort reranking algorithm.
    :param current_ranking_df: Pandas dataframe of ranking to be reranked.
    :param item_group_dict: Dictionary of items (keys) and their group membership (values).
    :param current_ranking_scores_df: Pandas dataframe of relevance scores associated with each item in the ranking.
    :param distribution: Dictionary of group proportions (groups are keys). Ex. [.5, .5] is a fifty-fifty split.
    :param k: Int, how long the returned ranking should be.
    :return: reranking, Pandas dataframe of items,item_group_reranked_dict, dictionary of items and group membership,  Pandas dataframe  of scores for reranking,
    """

    # Convert dataframes to numpy arrays
    current_ranking = current_ranking_df[0].to_numpy()
    current_group_ids = np.asarray([item_group_dict[i] for i in current_ranking])
    current_ranking_scores = current_ranking_scores_df[0].to_numpy()

    # score_list is <group id>  <score> and <rank> <startrank> <id>
    score_list = [
        (
            current_group_ids[i],
            current_ranking_scores[i],
            i + 1,
            i + 1,
            current_ranking[i],
        )
        for i in range(len(current_ranking))
    ]

    unique_group_ids = list(np.unique(current_group_ids))
    AttrScores = {}
    num_items_per_group = {}
    min_grp_count = {}
    GlobalAttrCounts = {}
    constructed_ranking_group_ids = {}  # []
    rankedScoreList = {}  # []
    maxIndices = {}  # []
    last_empty_indx = 0
    k_iter = 0

    for grp_id in unique_group_ids:
        num_items_per_group[grp_id] = 0
        min_grp_count[grp_id] = 0
        GlobalAttrCounts[grp_id] = sum([1 for elem in score_list if elem[0] == grp_id])
        AttrScores[grp_id] = [
            (item[1], item[0], item[2], item[3], item[4])
            for item in score_list
            if item[0] == grp_id
        ]  # to be initialized

    while last_empty_indx <= k:
        if last_empty_indx == len(score_list):
            break

        k_iter += 1
        temp_min_attr_count_for_curr_prefix = ddict(int)
        changedMins = {}
        for grp_id in unique_group_ids:
            temp_min_attr_count_for_curr_prefix[grp_id] = math.floor(
                k_iter * distribution[grp_id]
            )
            if (
                min_grp_count[grp_id] < temp_min_attr_count_for_curr_prefix[grp_id]
                and min_grp_count[grp_id] < GlobalAttrCounts[grp_id]
            ):
                changedMins[grp_id] = AttrScores[grp_id][num_items_per_group[grp_id]]

        if len(changedMins) != 0:
            ordChangedMins = sorted(
                changedMins.items(), key=lambda x: x[1][0], reverse=True
            )
            for item in ordChangedMins:
                constructed_ranking_group_ids[last_empty_indx] = item[0]
                rankedScoreList[last_empty_indx] = item[1]
                maxIndices[last_empty_indx] = k_iter
                start = last_empty_indx
                while (
                    start > 0
                    and maxIndices[start - 1] >= start
                    and rankedScoreList[start - 1][0] < rankedScoreList[start][0]
                ):
                    __swap(rankedScoreList, start - 1, start)
                    __swap(maxIndices, start - 1, start)
                    __swap(constructed_ranking_group_ids, start - 1, start)
                    start -= 1
                num_items_per_group[item[0]] += 1
                last_empty_indx += 1
                print('last_empty_indx', last_empty_indx)
            min_grp_count = dict(temp_min_attr_count_for_curr_prefix)

    K_items = []
    K_scores = []

    for i in range(0, k):
        item = rankedScoreList[i]
        K_items.append(item[4])
        K_scores.append(item[0])

    reranking = np.asarray(K_items)
    reranking_scores = np.asarray(K_scores)
    current_rank = list(current_ranking)
    reranking_ids = np.asarray(
        [current_group_ids[current_rank.index(item)] for item in reranking]
    )
    item_group_reranked_dict = dict(zip(reranking, reranking_ids))
    return (
        pd.DataFrame(reranking),
        item_group_reranked_dict,
        pd.DataFrame(reranking_scores),
    )


def __swap(temp_list, pos_i, pos_j):
    temp = temp_list[pos_i]
    temp_list[pos_i] = temp_list[pos_j]
    temp_list[pos_j] = temp
    

def EPSILONGREEDY(
    current_ranking_df, item_group_dict, current_ranking_scores_df, epsilon, seed
):
    """
    Epsilon-Greedy reranking algorithm.
    :param current_ranking_df: Pandas dataframe of ranking to be reranked.
    :param item_group_dict: Dictionary of items (keys) and their group membership (values).
    :param current_ranking_scores_df: Pandas dataframe of relevance scores associated with each item in the ranking.
    :param epsilon: Float epsilon value in [0,1].
    :param seed: Random seed value for reproducibility.
    :return: reranking, Pandas dataframe of items,item_group_reranked_dict, dictionary of items and group membership,  Pandas dataframe  of scores for reranking,
    """

    # Convert dataframes to numpy arrays
    current_ranking = current_ranking_df[0].to_numpy()
    current_group_ids = np.asarray([item_group_dict[i] for i in current_ranking])
    current_ranking_scores = current_ranking_scores_df[0].to_numpy()

    ranking = list(current_ranking)
    curr_ranking = copy.deepcopy(ranking)
    np.random.seed(seed)  # for reproducibility
    reranking = []
    for i in range(len(curr_ranking)):
        p = np.random.rand()
        if (
            p <= epsilon and i < len(curr_ranking) - 1
        ):  # swap items & can't swap last item
            temp = curr_ranking[i]
            j = np.random.randint(i + 1, len(curr_ranking))
            curr_ranking[i] = curr_ranking[j]
            curr_ranking[j] = temp
            reranking.append(curr_ranking[i])
        else:  # keep original ranking
            reranking.append(curr_ranking[i])

    reranking = np.asarray(reranking)
    reranking_scores = np.asarray(
        [current_ranking_scores[ranking.index(item)] for item in reranking]
    )
    reranking_ids = np.asarray(
        [current_group_ids[ranking.index(item)] for item in reranking]
    )
    item_group_reranked_dict = dict(zip(reranking, reranking_ids))
    return (
        pd.DataFrame(reranking),
        item_group_reranked_dict,
        pd.DataFrame(reranking_scores),
    )



## $\epsilon$-Greedy

In [38]:
stats = {}

In [27]:
stats['epsilon-greedy'] = {}

datasets = {'facebook':'/home/jrm28/fairness/in_processing_methods/NeuralCommonNeighbor/saved_output_new/facebook_puregcn_cn1_256_2_23220240920172321.pt', 
            'credit':'/home/jrm28/fairness/in_processing_methods/NeuralCommonNeighbor/saved_output_new/credit_puregcn_cn1_256_1_220240924145920.pt', 
            'german':'', 
            'nba':'', 
            'pokec_n':'', 
            'pokec_z':''}
           

for dataset, path in datasets:
    preds = torch.load(path, map_location=torch.device('cpu'))
    pred_rankings = pd.DataFrame(preds['test_preds'].argsort().numpy())
    preds_sens_values = dict(zip(range(len(pred_rankings)), preds['test_protected_groups'].numpy()))
    
    scores = pd.DataFrame(preds['test_preds'].numpy())
    
    # Applying the post-processing reranking algorithm
    reranking, item_group_reranked_dict, reranking_scores = EPSILONGREEDY(pred_rankings, preds_sens_values, scores, epsilon=0.1, seed=0)
    
    output = preds['test_preds']
    labels = preds['test_true']
    sens = preds['test_protected_groups']
    
    parity, equality = fair_metric(output.sigmoid().detach().numpy(), labels.numpy(), sens.numpy())
    parity, equality
    
    fair_topk = fairness_aware_topk(output.squeeze(), 100, sens, 'torch')
    fair_hitsk = fairness_aware_hits(output.detach()[labels.bool()].squeeze(), output.detach()[~labels.bool()].squeeze(), 100, torch.tensor(sens[labels.bool()]), 'torch')

    y_pred_pos = output.detach().numpy()[labels.bool()]
    y_pred_neg = output.detach().numpy()[~labels.bool()]
    hits = eval_hits(y_pred_pos, y_pred_neg, 100, 'numpy')
    precision = retrieval_precision(output.sigmoid().squeeze(), labels.bool(), 100).item()
    
    ndkl = NDKL(pd.DataFrame(output.sigmoid().squeeze().argsort().numpy()), dict(zip(range(output.shape[0]),sens.numpy())))

    stats['epsilon-greedy'][dataset] = {}
    stats['epsilon-greedy'][dataset]['parity'] = parity
    stats['epsilon-greedy'][dataset]['equality'] = equality
    stats['epsilon-greedy'][dataset]['fair_topk'] = fair_topk
    stats['epsilon-greedy'][dataset]['fair_hitsk'] = fair_hitsk
    stats['epsilon-greedy'][dataset] |= hits
    stats['epsilon-greedy'][dataset]['precision'] = precision
    stats['epsilon-greedy'][dataset]['ndkl'] = ndkl


In [20]:
NDKL(reranking, item_group_reranked_dict)

0.0024142595931737356

In [58]:
eval_hits(preds['test_preds'], preds['test_true'], 100, 'torch')

{'hits@100': 0.5434660684240045}

In [27]:
preds['test_preds']

tensor([ 3.5090,  3.5219,  3.6305,  ..., -7.4540, -7.4257, -5.7587])

In [31]:
ndcg_score([preds['test_true']], [preds['test_preds']], k=1000)

0.9929548637173663

In [21]:
ndcg_score([preds['test_true'][reranking.values.squeeze()]], [preds['test_preds'][reranking.values.squeeze()]], k=1000)

0.9929548637173663

In [35]:
K = 100
preds['test_true'][reranking[:K].values].sum() / K

tensor(0.0400)

In [13]:
reranking_scores

,0
0,3.508992
1,3.521893
2,3.630546
3,2.817092
4,3.910126
...,...
5344,3.331251
5345,3.643326
5346,3.515139
5347,3.595057


In [ ]:
'/home/jrm28/fairness/in_processing_methods/NeuralCommonNeighbor/saved_output_new/credit_puregcn_cn1_256_1_220240924145920.pt'

## FA*IR